In [1]:
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

In [2]:
outputPath1="./output1/"
serversRDD=sc.textFile("./data/Servers.txt")
appliedPatchesRDD=sc.textFile("./data/AppliedPatches.txt")

#Task 1

In [14]:
cleanedServersRDD=serversRDD.map(lambda x: (x.split(",")[0],x.split(",")[1]))

cleanedAppliedPatchesRDD=appliedPatchesRDD.map(lambda x: (x.split(",")[1],(x.split(",")[2].split("/")[0],1))) \
    .filter(lambda x: x[1][0]=='2022').reduceByKey(lambda a,b: (a[0],a[1]+b[1])).map(lambda x: (x[0],x[1][1]))

maxVal=cleanedAppliedPatchesRDD.values().max()

finalRDD=cleanedAppliedPatchesRDD.filter(lambda x: x[1]==maxVal).join(cleanedServersRDD).map(lambda x: (x[0],x[1][1]))

#Task 2

In [16]:
outputPath2="./output2/"
patchesRDD=sc.textFile("./data/Patches.txt")

In [36]:
cleanedPatchesRDD=patchesRDD.map(lambda x:(x.split(",")[1],x.split(",")[0])).groupByKey()

newCleanedAppliedPatchesRDD=appliedPatchesRDD.map(lambda x: (x.split(",")[1],x.split(",")[0])).groupByKey()

newJoinedRDD=cleanedServersRDD.join(newCleanedAppliedPatchesRDD).map(lambda x: (x[1][0],(x[0],x[1][1])))

def filtraggio(row):
  SID, patches = row
  patchesDone=set(patches[0])
  allPatches=set(patches[1])
  if patchesDone==allPatches:
    return SID


finalJoinRDD=newJoinedRDD.join(cleanedPatchesRDD).map(lambda x: (x[1][0][0],(x[1][0][1],x[1][1]))).map(filtraggio).filter(lambda x: x is not None)



In [37]:
finalJoinRDD.collect()

[]